# 3.1 怎麼彙整多個位置？


兩個位置各放一張數字卡，第一張 `[2,0]`，第二張 `[0,4]`。如果兩張各取一半，得到 `[1,2]`；如果我們此刻更需要第一張的資訊，就取第一張九成、第二張一成，結果應更靠近第一張。與 [2.2](https://birdhackor.github.io/tiny-perceptron-vlm/2.2.html) 把所有卡原封不動接起來不同，這次把多張卡混成一張，而且可以決定每張佔多少。

這種操作叫加權平均：每張向量乘上自己的比例，再逐欄相加。比例非負、加總為1，才能表示一份完整的分配。這與 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html) 的一般配方稍有區別：Linear的權重可以負、也不必加總為1；這裡先使用可解讀的讀取比例，讓結果成為各位置資訊的混合。向量與矩陣見 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

values = torch.tensor([[2.0, 0.0], [0.0, 4.0]])
weights = torch.tensor([0.9, 0.1])
output = weights @ values
print(output)
assert torch.allclose(output, torch.tensor([1.8, 0.4]))

輸入 `values` 是兩個位置、每位置兩個特徵，形狀 `[2,2]`；`weights` 是兩個讀取比例。`@` 完成加權平均，第一個輸出特徵為 `0.9×2+0.1×0=1.8`，第二個為 `0.9×0+0.1×4=0.4`。輸出只剩一張兩數的卡 `[1.8,0.4]`，位置軸已經加總掉，但特徵軸仍保留。並不是先把每張的特徵自己平均成一個數。

`torch.allclose` 逐格比較計算結果與手算值，容許浮點表示造成的小幅誤差，全部夠接近就回傳`True`。最後的`assert`在通過時沒有額外輸出；若結果不夠接近，就以`AssertionError`停下，提醒你檢查讀取比例、數值或手算結果。

這是注意力（attention）最核心的讀取動作：把不同位置的資訊按比例取回。它沒有真的把視線移到某一字，也不一定只選擇一個位置；例中兩張都參與，只是第一張多些。我們目前手工給0.9與0.1，下一節才問怎樣根據目前內容產生比例。先分開「如何混合」與「如何決定混合比例」，就能避免把整套機制當成一個黑盒名字。

注意力輸出也不告訴你究竟哪張卡包含什麼語意。這兩個特徵只是為了算得清楚而指定的數字；真實模型中的特徵由訓練學習，一張卡可能同時記錄多種資訊。讀取比例較高，只能說對這次混合貢獻較大，不能單靠它證明模型理解了某個概念。

如果這兩個比例都是0.5，結果就在兩卡中間；若為1、0，結果等於第一張。這些端點能幫你核對加權平均的方向是否寫對。對所有位置同樣平均也是一種讀法，但沒有能力依不同問題改讀取重點。

練習把`weights = torch.tensor([0.9, 0.1])`內的兩個比例交換成`[0.1,0.9]`，並把檢查的預期輸出改成你手算的值。先算應為 `[0.2,3.6]`，再執行核對；結果應更靠近第二張。你改的是讀法，不是兩張卡的內容，這個區別會貫穿後面Q、K、V的拆解。
